# TV Character Chatbot — Fine-tuning with QLoRA

Fine-tunes **SmolLM2 1.7B Instruct** on dialogue from *The Office* and *The Big Bang Theory*
using **QLoRA** (4-bit NF4 quantisation + LoRA adapters).

### What this notebook does
1. Builds `(context → character_response)` training pairs from the show CSVs
2. Formats them using SmolLM2's chat template
3. Loads the base model at 4-bit precision with `bitsandbytes`
4. Wraps it with LoRA adapters (`peft`) — only ~1% of parameters are trained
5. Trains with `trl.SFTTrainer` and saves the adapters

### Requirements
- GPU with ≥8 GB VRAM (or Apple Silicon — model is only ~3.4 GB in fp16)
- No HuggingFace token needed — `HuggingFaceTB/SmolLM2-1.7B-Instruct` is publicly accessible

In [ ]:
# Install dependencies (run once)
!pip install -q \
    torch transformers accelerate \
    peft trl bitsandbytes \
    datasets pandas

In [ ]:
# SmolLM2-1.7B-Instruct is publicly available — no HuggingFace token required.
print("No authentication needed for HuggingFaceTB/SmolLM2-1.7B-Instruct. Ready to proceed.")

In [ ]:
import os
from typing import Dict, List

import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from trl import SFTTrainer, SFTConfig

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Configuration
Adjust these variables to control the training run.

In [ ]:
CFG = {
    # Model — publicly accessible, no token required
    'model_id'      : 'HuggingFaceTB/SmolLM2-1.7B-Instruct',
    'output_dir'    : './finetuned_model',

    # Data
    'office_csv'    : 'TheOffice.csv',
    'bigbang_csv'   : 'TBBTcleaned.csv',
    'context_turns' : 8,       # preceding turns of dialogue used as prompt context (plan: 6-12)
    'max_samples'   : None,    # None = use all Sheldon lines (~3-4K after filtering)

    # Characters to train on
    'characters'    : ['Sheldon'],

    # LoRA
    'lora_r'        : 16,
    'lora_alpha'    : 32,

    # Training
    'epochs'        : 2,
    'batch_size'    : 2,
    'grad_accum'    : 8,       # effective batch = 16 (2 × 8)
    'lr'            : 2e-4,
    'max_seq_len'   : 256,     # increase to 512 on CUDA for quality

    # Dataset split — by episode to avoid leakage (same scene in train + eval)
    # Hold out the last val_frac of unique (season, episode) pairs per show as validation.
    'val_frac'      : 0.10,    # 10 % of episodes → chronological hold-out
}

## Character Metadata
System-prompt descriptions and show membership for each character.

In [ ]:
CHARACTER_DESCRIPTIONS: Dict[str, str] = {
    # ── The Office ───────────────────────────────────────────────────────────
    'Michael': (
        'You are Michael Scott, the Regional Manager of Dunder Mifflin Scranton. '
        'You are enthusiastic, cringe-worthy, and desperately want to be liked. '
        'You believe you are everyone\'s best friend and the world\'s greatest boss. '
        'You often say \'That\'s what she said\' and reference movies inappropriately.'
    ),
    'Dwight': (
        'You are Dwight Schrute, Assistant (to the) Regional Manager at Dunder Mifflin. '
        'You are intensely loyal, own a beet farm, are a volunteer sheriff\'s deputy. '
        'You take everything with extreme seriousness and believe you are destined for greatness.'
    ),
    'Jim': (
        'You are Jim Halpert, a salesman at Dunder Mifflin Scranton. '
        'You are sardonic, laid-back, and love pranking Dwight. '
        'You react to absurdity with a knowing glance and dry wit.'
    ),
    'Pam': (
        'You are Pam Beesly, the receptionist at Dunder Mifflin. '
        'You are kind, artistic, quietly funny, and often the voice of reason.'
    ),
    'Andy': (
        'You are Andy Bernard, a salesman at Dunder Mifflin who attended Cornell '
        '(which you mention constantly). You love a cappella singing and are eager to impress.'
    ),
    'Ryan': (
        'You are Ryan Howard, who started as a temp at Dunder Mifflin. '
        'You are self-absorbed, ambitious, and constantly reinventing your image.'
    ),
    'Kevin': (
        'You are Kevin Malone, an accountant at Dunder Mifflin. '
        'You are simple-minded, lovable, and passionate about food, poker, and your band Scrantonicity.'
    ),
    'Angela': (
        'You are Angela Martin, head of accounting at Dunder Mifflin. '
        'You are uptight, judgmental, and obsessively devoted to your cats.'
    ),

    # ── The Big Bang Theory ──────────────────────────────────────────────────
    'Sheldon': (
        'You are Sheldon Cooper, a theoretical physicist at Caltech with an IQ of 187. '
        'You lack social awareness, follow rigid routines, and have a designated spot on the couch. '
        'You believe you are intellectually superior to everyone. '
        'You only say \'Bazinga!\' on rare occasions — specifically when you reveal that a previous '
        'statement was a deliberate deception or joke. '
        'Never use \'Bazinga!\' as a general exclamation, never end a response with it, '
        'and never use it more than once per conversation.'
    ),
    'Leonard': (
        'You are Leonard Hofstadter, an experimental physicist at Caltech. '
        'You are insecure, romantic, and constantly caught between Sheldon\'s demands '
        'and trying to live a normal life.'
    ),
    'Penny': (
        'You are Penny, an aspiring actress working as a waitress at The Cheesecake Factory. '
        'You are street-smart, fun, and perpetually bemused by your genius neighbors.'
    ),
    'Howard': (
        'You are Howard Wolowitz, an aerospace engineer at Caltech and NASA astronaut. '
        'You are a self-proclaimed ladies\' man and a mama\'s boy who still lives with his mother.'
    ),
    'Raj': (
        'You are Raj Koothrappali, an astrophysicist at Caltech from New Delhi. '
        'You cannot speak to women unless you have consumed alcohol. '
        'You are deeply sentimental, romantic, and love Bollywood.'
    ),
    'Bernadette': (
        'You are Bernadette Rostenkowski-Wolowitz, a microbiologist and Howard\'s wife. '
        'You have a sweet, high-pitched voice that belies a fierce, no-nonsense personality.'
    ),
    'Amy': (
        'You are Amy Farrah Fowler, a neurobiologist and Sheldon\'s girlfriend. '
        'You desperately crave normal friendships despite your painfully awkward academic demeanor.'
    ),
}

SHOW_MAP: Dict[str, str] = {
    'Michael': 'The Office',    'Dwight': 'The Office',
    'Jim': 'The Office',        'Pam': 'The Office',
    'Andy': 'The Office',       'Ryan': 'The Office',
    'Kevin': 'The Office',      'Angela': 'The Office',
    'Sheldon': 'The Big Bang Theory',   'Leonard': 'The Big Bang Theory',
    'Penny': 'The Big Bang Theory',     'Howard': 'The Big Bang Theory',
    'Raj': 'The Big Bang Theory',       'Bernadette': 'The Big Bang Theory',
    'Amy': 'The Big Bang Theory',
}

print(f'Loaded descriptions for {len(CHARACTER_DESCRIPTIONS)} characters')

## Data Preparation

For each scene we create `(preceding_dialogue → character_response)` pairs.
Up to `context_turns` lines before a character speaks become the user turn;
the character's line is the assistant turn.

```
System : You are Sheldon Cooper...
User   : Leonard: You okay?
         Penny: He looks fine to me.
         Now respond as Sheldon.
Assistant: I'm not fine. I'm never fine. 'Fine' is a word people use when
           they don't want to discuss what's actually wrong.
```

In [ ]:
import re

def _norm_text(val: object) -> str:
    if pd.isna(val):
        return ''
    return str(val).strip()

def _clean_dialogue(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = re.sub(r'[\"\']+', '', text)
    text = re.sub(r',{2,}', ',', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

def _strip_parentheticals(name: str) -> str:
    return re.sub(r'\s*\(.*?\)', '', name).strip()

def _merge_consecutive(lines: List[tuple]) -> List[tuple]:
    """Merge consecutive utterances by the same speaker into one turn."""
    if not lines:
        return []
    merged = [[lines[0][0], lines[0][1]]]
    for speaker, dialogue in lines[1:]:
        if speaker == merged[-1][0]:
            merged[-1][1] = merged[-1][1].rstrip() + ' ' + dialogue
        else:
            merged.append([speaker, dialogue])
    return [(s, d) for s, d in merged]

_BBT_NON_CHARS  = {'scene', 'stage direction', 'all'}
_BBT_CHAR_ALIASES = {
    'Beverley': 'Beverly', 'Lesley': 'Leslie',
    'Dr Koothrappali': 'Raj', 'Dr Hofstadter': 'Leonard',
    'Past Sheldon': 'Sheldon', 'Past Leonard': 'Leonard',
    'Mrs Cooper': 'Mary',
}


def build_office_pairs(csv_path: str, target_chars: List[str], context_turns: int) -> List[Dict]:
    df = pd.read_csv(csv_path)
    df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
    df['speaker'] = df['speaker'].fillna('').str.strip()
    df['line']    = df['line'].fillna('').apply(_clean_dialogue)
    df = df[(df['speaker'] != '') & (df['line'] != '')]

    examples: List[Dict] = []
    for (season, episode, scene), group in df.groupby(['season', 'episode', 'scene'], sort=True):
        lines  = list(zip(group['speaker'].tolist(), group['line'].tolist()))
        merged = _merge_consecutive(lines)
        for i, (speaker, line) in enumerate(merged):
            if speaker not in target_chars:
                continue
            ctx = merged[max(0, i - context_turns):i]
            if not ctx:
                continue
            examples.append({
                'character'    : speaker,
                'season'       : int(season),
                'episode'      : int(episode),
                'scene_setting': '',
                'context'      : '\n'.join(f'{s}: {l}' for s, l in ctx),
                'response'     : line,
            })
    return examples


def build_bigbang_pairs(csv_path: str, target_chars: List[str], context_turns: int) -> List[Dict]:
    """Parse TBBTcleaned.csv which uses an explicit scene_id column.

    Column schema: season, episode, character, dialogue, scene_id
    """
    df = pd.read_csv(csv_path)

    # Normalise character names
    df['character'] = (
        df['character'].fillna('').str.strip()
        .apply(_strip_parentheticals)
        .map(lambda c: _BBT_CHAR_ALIASES.get(c, c))
    )
    df['dialogue'] = df['dialogue'].fillna('').apply(_clean_dialogue)

    # Drop non-character rows (Scene markers, Stage Directions, etc.)
    df = df[~df['character'].str.lower().isin(_BBT_NON_CHARS)]
    df = df[(df['character'] != '') & (df['dialogue'] != '')]

    examples: List[Dict] = []

    # scene_id already encodes scene boundaries — group directly
    for (season, episode, scene_id), group in df.groupby(['season', 'episode', 'scene_id'], sort=True):
        lines  = list(zip(group['character'].tolist(), group['dialogue'].tolist()))
        merged = _merge_consecutive(lines)
        for i, (speaker, line) in enumerate(merged):
            if speaker not in target_chars:
                continue
            ctx = merged[max(0, i - context_turns):i]
            if not ctx:
                continue
            examples.append({
                'character'    : speaker,
                'season'       : int(season),
                'episode'      : int(episode),
                'scene_setting': '',
                'context'      : '\n'.join(f'{s}: {l}' for s, l in ctx),
                'response'     : line,
            })
    return examples

In [ ]:
def format_chat(example: Dict, tokenizer) -> str:
    char    = example['character']
    show    = SHOW_MAP.get(char, 'a TV show')
    desc    = CHARACTER_DESCRIPTIONS.get(char, f'You are {char}.')
    setting = example.get('scene_setting', '')
    system  = f'{desc} Stay fully in character as {char} from {show}.'

    # Include scene location when available so the model learns scene-aware responses
    setting_line = f'\n[Scene: {setting}]' if setting else ''
    user_content = (
        f'Here is some dialogue context:{setting_line}\n'
        f'{example["context"]}\n\n'
        f'Now respond as {char}.'
    )

    messages = [
        {'role': 'system',    'content': system},
        {'role': 'user',      'content': user_content},
        {'role': 'assistant', 'content': example['response']},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

## Load Tokenizer & Build Dataset

In [ ]:
print(f'Loading tokenizer: {CFG["model_id"]}')
tokenizer = AutoTokenizer.from_pretrained(CFG['model_id'])
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'
print('Tokenizer loaded.')

In [ ]:
office_chars  = [c for c in CFG['characters'] if SHOW_MAP.get(c) == 'The Office']
bigbang_chars = [c for c in CFG['characters'] if SHOW_MAP.get(c) == 'The Big Bang Theory']

raw: List[Dict] = []

if office_chars and os.path.exists(CFG['office_csv']):
    pairs = build_office_pairs(CFG['office_csv'], office_chars, CFG['context_turns'])
    raw.extend(pairs)
    print(f'The Office      : {len(pairs):,} pairs  {office_chars}')

if bigbang_chars and os.path.exists(CFG['bigbang_csv']):
    pairs = build_bigbang_pairs(CFG['bigbang_csv'], bigbang_chars, CFG['context_turns'])
    raw.extend(pairs)
    print(f'Big Bang Theory : {len(pairs):,} pairs  {bigbang_chars}')

if not raw:
    raise RuntimeError('No training examples generated — check CSV paths.')

print(f'\nTotal pairs     : {len(raw):,}')

# ── Episode-based chronological split ───────────────────────────────────────
# Group by actual (season, episode) keys so the same scene never appears in
# both train and val — random row-level splits leak context turns across the
# boundary when consecutive rows from one episode land on opposite sides.

def _episode_split(examples: List[Dict], val_frac: float = 0.10):
    """
    Hold out the last ``val_frac`` of unique (season, episode) pairs as validation.

    Examples must be ordered chronologically (build_*_pairs uses sort=True).
    Returns (train_examples, val_examples).
    """
    if not examples:
        return examples, []

    # Build ordered list of unique episode keys (preserves chronological order)
    seen: set = set()
    ep_order: List[tuple] = []
    for ex in examples:
        key = (ex['season'], ex['episode'])
        if key not in seen:
            seen.add(key)
            ep_order.append(key)

    n_val_eps = max(1, int(len(ep_order) * val_frac))
    val_eps   = set(ep_order[-n_val_eps:])

    train = [ex for ex in examples if (ex['season'], ex['episode']) not in val_eps]
    val   = [ex for ex in examples if (ex['season'], ex['episode']) in val_eps]
    return train, val


raw_bbt    = [ex for ex in raw if SHOW_MAP.get(ex['character']) == 'The Big Bang Theory']
raw_office = [ex for ex in raw if SHOW_MAP.get(ex['character']) == 'The Office']

train_bbt,    val_bbt    = _episode_split(raw_bbt,    CFG['val_frac'])
train_office, val_office = _episode_split(raw_office, CFG['val_frac'])
train_raw = train_bbt + train_office

# Report how many unique episodes landed in each split
val_eps_bbt    = len({(ex['season'], ex['episode']) for ex in val_bbt})
val_eps_office = len({(ex['season'], ex['episode']) for ex in val_office})
print(f'Train examples  : {len(train_raw):,}')
print(f'Val examples    : {len(val_bbt) + len(val_office):,}'
      f'  (BBT: {val_eps_bbt} eps | Office: {val_eps_office} eps — last {int(CFG["val_frac"]*100)}%, chronological)')

# ── Format using SmolLM2 chat template ──────────────────────────────────────
formatted_train = [format_chat(ex, tokenizer) for ex in train_raw]
dataset         = Dataset.from_dict({'text': formatted_train})

# Subsample if configured (useful for quick iteration on MPS)
if CFG.get('max_samples') and CFG['max_samples'] < len(dataset):
    dataset = dataset.select(range(CFG['max_samples']))
    print(f'Subsampled to   : {len(dataset):,} examples')

print(f'Final dataset   : {len(dataset):,} training examples')
print('\nSample formatted example:')
print(dataset[0]['text'][:600], '...')

## Load Model

SmolLM2-1.7B-Instruct is only **~3.4 GB in fp16**, so device strategy adapts automatically:

| Device | Strategy | VRAM / RAM |
|---|---|---|
| **CUDA GPU** | 4-bit NF4 quantisation | ~1.1 GB VRAM |
| **Apple Silicon (MPS)** | fp16, loaded to MPS | ~3.4 GB Unified Memory |
| **CPU** | fp16 | ~3.4 GB RAM |

On CUDA, `BitsAndBytesConfig` with double NF4 quantisation is used.  
On Apple Silicon, the model fits comfortably without quantisation — `bitsandbytes` is a CUDA-only library.

In [ ]:
# Device detection — bitsandbytes 4-bit quantisation is CUDA-only
def _detect_device() -> str:
    if torch.cuda.is_available():
        return 'cuda'
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return 'mps'
    return 'cpu'

device = _detect_device()
print(f'Device: {device}')

if device == 'cuda':
    # 4-bit NF4 — reduces VRAM from ~3.4 GB (fp16) to ~1.1 GB
    bnb_config = BitsAndBytesConfig(
        load_in_4bit              = True,
        bnb_4bit_use_double_quant = True,
        bnb_4bit_quant_type       = 'nf4',
        bnb_4bit_compute_dtype    = torch.bfloat16,
    )
    print(f'Loading model: {CFG["model_id"]} (4-bit NF4) ...')
    model = AutoModelForCausalLM.from_pretrained(
        CFG['model_id'],
        quantization_config = bnb_config,
        device_map          = 'auto',
        torch_dtype         = torch.bfloat16,
    )
    model = prepare_model_for_kbit_training(model)
    print('Base model loaded in 4-bit NF4, prepared for k-bit training.')
else:
    # Apple Silicon (MPS) or CPU — load in fp16 directly
    # SmolLM2-1.7B is only ~3.4 GB in fp16 and fits in Apple Unified Memory
    dtype = torch.float16
    print(f'Loading model: {CFG["model_id"]} (fp16, {device}) ...')
    model = AutoModelForCausalLM.from_pretrained(
        CFG['model_id'],
        torch_dtype = dtype,
    )
    model = model.to(device)
    print(f'Base model loaded in fp16 on {device}.')

In [ ]:
lora_cfg = LoraConfig(
    r              = CFG['lora_r'],
    lora_alpha     = CFG['lora_alpha'],
    target_modules = [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',   # attention
        'gate_proj', 'up_proj', 'down_proj',        # MLP (SwiGLU)
    ],
    lora_dropout   = 0.05,
    bias           = 'none',
    task_type      = 'CAUSAL_LM',
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
# Expected: ~0.8% of total params (≈ 13M / 1.7B) with r=16 across 7 modules × 28 layers

## Training

`SFTConfig` is used instead of `TrainingArguments` — it is a drop-in superset that accepts
`max_seq_length` and `dataset_text_field` natively, avoiding the deprecation warning when
passing them directly to `SFTTrainer`.

**Speed levers** (all adjustable in `CFG`):

| Setting | Default | Effect |
|---|---|---|
| `max_samples` | 3000 | Caps training examples — biggest time saver on MPS |
| `max_seq_len` | 256 | Shorter sequences = faster steps (512 → ~4× slower) |
| `batch_size` | 2 | Lower = less RAM pressure on Apple Silicon |
| `grad_accum` | 8 | Keeps effective batch = 16 |
| `epochs` | 2 | Reduce to 1 for a quick test run |

| Setting | CUDA | MPS / CPU |
|---|---|---|
| Precision | `bf16` | `fp16` |
| Optimiser | `paged_adamw_8bit` | `adamw_torch` |

In [ ]:
# SFTConfig = TrainingArguments + SFT-specific params (max_seq_length, dataset_text_field, etc.)
# Using SFTConfig avoids the DeprecationWarning from passing max_seq_length to SFTTrainer directly.
training_args = SFTConfig(
    # ── SFT-specific ────────────────────────────────────────────────────────
    max_seq_length              = CFG['max_seq_len'],
    dataset_text_field          = 'text',

    # ── Output & logging ────────────────────────────────────────────────────
    output_dir                  = CFG['output_dir'],
    logging_steps               = 25,
    save_steps                  = 200,
    save_total_limit            = 2,
    report_to                   = 'none',

    # ── Training schedule ───────────────────────────────────────────────────
    num_train_epochs            = CFG['epochs'],
    per_device_train_batch_size = CFG['batch_size'],
    gradient_accumulation_steps = CFG['grad_accum'],   # effective batch = batch_size × grad_accum
    warmup_steps                = 50,
    learning_rate               = CFG['lr'],
    lr_scheduler_type           = 'cosine',

    # ── Precision & optimiser (device-aware) ────────────────────────────────
    bf16                        = (device == 'cuda'),   # bfloat16 — CUDA (Ampere+) only
    fp16                        = (device == 'mps'),    # float16  — Apple Silicon
    optim                       = 'paged_adamw_8bit' if device == 'cuda' else 'adamw_torch',
)

trainer = SFTTrainer(
    model         = model,
    train_dataset = dataset,
    args          = training_args,
)

print(f'Starting fine-tuning on {device} — {len(dataset):,} examples × {CFG["epochs"]} epochs...')
trainer.train()

In [ ]:
print(f'Saving LoRA adapters → {CFG["output_dir"]}')
trainer.model.save_pretrained(CFG['output_dir'])
tokenizer.save_pretrained(CFG['output_dir'])
print('Done! Load with: PeftModel.from_pretrained(base_model, CFG["output_dir"])')

## Optional: Quick Inference Test

Verify the fine-tuned adapters produce in-character responses.

In [ ]:
from peft import PeftModel

def chat_with_character(character: str, user_msg: str, max_new_tokens: int = 200) -> str:
    show = SHOW_MAP.get(character, 'a TV show')
    desc = CHARACTER_DESCRIPTIONS.get(character, f'You are {character}.')
    system = f'{desc} Stay fully in character as {character} from {show}.'

    messages = [
        {'role': 'system', 'content': system},
        {'role': 'user',   'content': user_msg},
    ]

    # return_dict=True ensures we get a BatchEncoding dict with both
    # input_ids and attention_mask — works consistently across tokenizer implementations
    tokenized = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        return_tensors        = 'pt',
        return_dict           = True,
    )
    input_ids      = tokenized['input_ids'].to(model.device)
    attention_mask = tokenized['attention_mask'].to(model.device)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            attention_mask     = attention_mask,
            max_new_tokens     = max_new_tokens,
            temperature        = 0.8,
            top_p              = 0.9,
            do_sample          = True,
            repetition_penalty = 1.15,
            pad_token_id       = tokenizer.eos_token_id,
        )
    reply_ids = output[0][input_ids.shape[-1]:]
    return tokenizer.decode(reply_ids, skip_special_tokens=True).strip()


# Test a few characters
tests = [
    ('Sheldon', 'What do you think about string theory?'),
    ('Dwight',  'What is your greatest achievement?'),
    ('Michael', 'How do you motivate your team?'),
]

for char, msg in tests:
    print(f'\n--- {char} ---')
    print(f'User     : {msg}')
    print(f'{char:8} : {chat_with_character(char, msg)}')